In [1]:
import time
from datetime import datetime, timedelta
import os
import numpy as np
import pandas as pd
import requests


from web3 import Web3

from dotenv import load_dotenv
load_dotenv()

True

# Сбор данных

Исторические данные о priority fees из BigQuery:
```
with blocks as (
  SELECT timestamp, 
       number as height, 
      --  `hash`,
      --  size,
      --  gas_limit,
      --  gas_used,
       transaction_count,
       base_fee_per_gas
  FROM `bigquery-public-data.crypto_ethereum.blocks` 
  WHERE TIMESTAMP_TRUNC(timestamp, DAY) >= TIMESTAMP("2025-01-01") 
    and TIMESTAMP_TRUNC(timestamp, DAY) < TIMESTAMP("2026-02-16") 
    -- WHERE TIMESTAMP_TRUNC(timestamp, DAY) = TIMESTAMP("2026-04-13") 
)

select block_timestamp, 
       block_number, 
      --  max(transaction_count) as transaction_count,
      --  max(base_fee_per_gas) as base_fee_per_gas,
       min(priority_fee) AS priority_min,

       APPROX_QUANTILES(priority_fee, 100)[OFFSET(50)] AS priority_p50,
       APPROX_QUANTILES(priority_fee, 100)[OFFSET(90)] AS priority_p90,
       APPROX_QUANTILES(priority_fee, 100)[OFFSET(95)] AS priority_p95,

       MAX(priority_fee) AS priority_max
from (
  SELECT 
        block_timestamp,
        block_number, 
        t.`hash` as txn_hash,
        receipt_effective_gas_price - blocks.base_fee_per_gas as priority_fee,
        -- receipt_effective_gas_price,
        -- blocks.base_fee_per_gas as base_fee_per_gas,
        -- max_fee_per_gas,
        -- max_priority_fee_per_gas,
        -- transaction_count
  FROM `bigquery-public-data.crypto_ethereum.transactions` t
  left join blocks on blocks.height = t.block_number
  WHERE
  TIMESTAMP_TRUNC(block_timestamp, DAY) >= TIMESTAMP("2025-01-01") 
    and TIMESTAMP_TRUNC(block_timestamp, DAY) < TIMESTAMP("2026-02-16") 
  -- TIMESTAMP_TRUNC(block_timestamp, DAY) = TIMESTAMP("2026-04-13") 
)
group by block_timestamp, block_number
```

In [2]:
df = pd.read_csv('data/priority_fees_2025-01-01_2026-02-15.csv')
df

,block_timestamp,block_number,priority_min,priority_p50,priority_p90,priority_p95,priority_max
0,2025-05-13 01:43:23 UTC,22471117,0,500000000,2000000000,4073000000,79345040579
1,2025-05-13 10:44:11 UTC,22473788,0,656206698,2000000000,3158501495,150238968729
2,2025-05-09 09:35:11 UTC,22445059,0,4950509445,22171114152,31815414700,149348908330
3,2025-07-01 13:18:35 UTC,22824637,0,1187345095,2000000000,2387198373,30000000000
4,2025-07-18 02:08:23 UTC,22943004,0,1580580151,4000000000,5020000000,73318893084
...,...,...,...,...,...,...,...
2937983,2025-03-19 00:39:23 UTC,22077393,0,101001040,2646725491,8000000000,73709527913
2937984,2025-05-29 22:47:47 UTC,22591474,0,970000000,3000000000,8000000000,88000000000
2937985,2025-02-25 06:59:47 UTC,21921709,0,567701423,3324912588,8000000000,89205828168
2937986,2025-03-09 16:44:59 UTC,22010585,0,970000000,2991760751,8000000000,43493381039


Добавим признаки, относящиеся только к блокам:

In [3]:
df_blocks = pd.read_csv('data/eth_blocks_2025-01-01_2026-02-15.csv')
df_blocks

,timestamp,height,hash,size,gas_limit,gas_used,transaction_count,base_fee_per_gas
0,2026-02-15 00:32:35 UTC,24458637,0x173fa6c45072b36b3b2489bf945c813f5eeb2fb7b7ce...,123436,59999886,17391909,108,35527943
1,2026-02-15 16:49:47 UTC,24463507,0xe0c7adb0bd468c81f23eb23627e60e5e373f1e1850da...,136110,59941351,24197479,272,46609559
2,2026-02-15 05:17:23 UTC,24460056,0xaf0bfcf9fecc2e7999c498184bbb9a36c50fcddd59a4...,206574,59941237,52959002,166,46207812
3,2026-02-15 14:31:35 UTC,24462819,0x8e9a8703ee7c65965eb2c4717ca74f9c48442f5d2542...,151227,59999886,42068893,263,33619778
4,2026-02-15 07:24:23 UTC,24460690,0xdca86811dbd9c17587a3b01bf72e16f1d633bc9d6ccb...,170208,59999886,46730062,481,48904977
...,...,...,...,...,...,...,...,...
2939758,2026-01-16 00:49:11 UTC,24243802,0xe03f9bf772619c82fc0a37bd64d25bbece1bbc3b9033...,236213,60000000,58358979,1156,40700653
2939759,2026-01-16 23:17:47 UTC,24250526,0xc3d889f810a1730f34473aaf4e6d463295d9180da2ee...,575739,60000000,59985849,178,29792259
2939760,2026-01-16 20:21:23 UTC,24249648,0x090cb4a592df450e965f775cbefd085620312cc003c4...,63704,60000000,12596083,154,50460437
2939761,2026-01-16 00:19:35 UTC,24243655,0xfc34e3be6633a3849bcc5e6155cf21ead59165682509...,83523,60000000,20000456,159,48073799


In [6]:
blocks_priority = (
    df[['block_timestamp', 'block_number', 'priority_p50', 'priority_p90', 'priority_p95', 'priority_max']]
    .merge(
        df_blocks[['height', 'gas_limit', 'gas_used', 'transaction_count', 'base_fee_per_gas']],
        left_on='block_number',
        right_on='height',
        how='inner',
    )
    .rename(columns={'block_timestamp': 'timestamp', 'block_number': 'height'})
)

Добавим исторические данные о цене ETH:

In [8]:
eth_prices = pd.read_csv("data/eth_minutes_2025_2026.csv")
eth_prices

,UNIT,TIMESTAMP,open,high,low,close,volume,QUOTE_VOLUME,VOLUME_TOP_TIER,QUOTE_VOLUME_TOP_TIER,VOLUME_DIRECT,QUOTE_VOLUME_DIRECT,VOLUME_TOP_TIER_DIRECT,QUOTE_VOLUME_TOP_TIER_DIRECT,datetime
0,MINUTE,1735636080,3378.569323,3380.485797,3378.217773,3380.465130,1767.621519,5.974709e+06,969.995245,3.278723e+06,88.282421,298298.946808,87.683491,296273.130839,2024-12-31 09:08:00
1,MINUTE,1735636140,3380.465130,3382.087644,3379.523548,3381.908966,2500.374313,8.453734e+06,1622.858775,5.487008e+06,133.230708,450299.577347,132.825178,448927.966293,2024-12-31 09:09:00
2,MINUTE,1735636200,3381.908966,3383.238625,3381.428658,3382.847440,2221.012152,7.513716e+06,1247.153641,4.219135e+06,188.263742,636652.513877,184.458903,623698.905942,2024-12-31 09:10:00
3,MINUTE,1735636260,3382.847440,3383.516298,3382.149028,3382.240800,1515.060300,5.125635e+06,867.835997,2.936108e+06,85.229725,288205.770504,85.036495,287551.856548,2024-12-31 09:11:00
4,MINUTE,1735636320,3382.240800,3383.575566,3381.438189,3381.591191,1391.147431,4.706566e+06,819.917387,2.773771e+06,78.714229,266118.741873,78.559561,265595.298606,2024-12-31 09:12:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
593988,MINUTE,1771275360,1993.592348,1993.592436,1991.998667,1992.059820,2449.901777,4.879452e+06,1042.716522,2.076631e+06,169.015468,336451.804526,162.860668,324202.884306,2026-02-16 20:56:00
593989,MINUTE,1771275420,1992.059820,1993.146110,1991.808684,1992.881738,2614.963938,5.208310e+06,1282.120863,2.553640e+06,188.397666,375039.166557,187.591856,373435.571706,2026-02-16 20:57:00
593990,MINUTE,1771275480,1992.881738,1993.668483,1992.616035,1993.666354,2378.978058,4.741151e+06,1151.227796,2.294991e+06,157.786077,314198.056788,157.553098,313734.635499,2026-02-16 20:58:00
593991,MINUTE,1771275540,1993.666354,1994.365204,1993.646374,1993.967930,3014.687517,6.009953e+06,1534.545144,3.059086e+06,222.362850,443097.765103,218.906048,436208.518704,2026-02-16 20:59:00


In [9]:
blocks_priority['timestamp'] = pd.to_datetime(blocks_priority['timestamp']).dt.tz_localize(None) # убираем timezone, чтобы не было проблем при объединении с ценами
eth_prices['datetime'] = pd.to_datetime(eth_prices['datetime']).dt.tz_localize(None)

In [10]:
blocks_priority = blocks_priority.sort_values('timestamp')
eth_prices = eth_prices.sort_values('datetime')

In [11]:
priority_blocks_with_price = pd.merge_asof(
    blocks_priority,
    eth_prices[['datetime', 'close']],
    left_on='timestamp',
    right_on='datetime',
    direction='backward' # чтобы избежать утечки будущего, всегда берем последнее известное значение цены
)

In [13]:
priority_blocks_with_price.rename(columns={"close": "last_eth_price"}, inplace=True)

In [15]:
priority_blocks_with_price.drop(columns=['datetime'], inplace=True)

In [16]:
priority_blocks_with_price.to_csv("data/priority_blocks_with_price.csv", index=False)

# Feature engineering